# WM-811K validation-selected 앙상블 고정 test

Validation에서 확정한 `baseline 50% + robust 50%`, none bias `0.1`, seed `42`를 변경하지 않고 **test를 단 한 번** 평가합니다. 현재 배포 중인 실제 checkpoint와 직접 비교하며, 결과를 보고 파라미터를 다시 조정하지 않습니다. 같은 출력 폴더에 결과가 이미 있으면 추론을 반복하지 않습니다.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/AI_project/WM811K_robust_hierarchical/validation_multiseed')
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_robust_hierarchical/locked_test_ensemble_0_5_bias_0_1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in (DATA_DIR/'wafer_maps_64.npy', DATA_DIR/'labels.npy', EXPERIMENT_ROOT/'runs'):
    assert path.exists(), f'필수 파일이 없습니다: {path}'


## 로컬 파일 5개를 차례로 선택

1. `train_wm811k_cnn.py`
2. `train_wm811k_robust_hierarchical_candidates.py`
3. `evaluate_wm811k_robust_ensemble_locked_test.py`
4. `split_assignments.csv`
5. 현재 배포 모델인 `결과물/wm811k/selected_model_results/best_model.pt`


In [ ]:
def upload_as(target_name):
    print(f'{target_name} 파일을 선택하세요.')
    uploaded = files.upload()
    assert len(uploaded) == 1, '이번 선택창에서는 파일 하나만 선택하세요.'
    _, content = next(iter(uploaded.items()))
    target = Path('/content') / target_name
    target.write_bytes(content)
    return target

BASE_SCRIPT = upload_as('train_wm811k_cnn.py')
ROBUST_SCRIPT = upload_as('train_wm811k_robust_hierarchical_candidates.py')
TEST_SCRIPT = upload_as('evaluate_wm811k_robust_ensemble_locked_test.py')
ASSIGNMENTS = upload_as('split_assignments.csv')
DEPLOYED_CHECKPOINT = upload_as('deployed_best_model.pt')


## 고정 test 실행

이 셀부터 test가 처음 열립니다. 선택값은 코드에 고정되어 있으며 재탐색하지 않습니다. T4에서 대략 5~10분 걸립니다.


In [ ]:
command = [
    sys.executable, str(TEST_SCRIPT),
    '--maps', str(DATA_DIR/'wafer_maps_64.npy'),
    '--labels', str(DATA_DIR/'labels.npy'),
    '--assignments', str(ASSIGNMENTS),
    '--experiment-root', str(EXPERIMENT_ROOT),
    '--deployed-checkpoint', str(DEPLOYED_CHECKPOINT),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', '512', '--num-workers', '2',
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'고정 test 평가 실패: exit code {return_code}'


In [ ]:
summary = json.loads((OUTPUT_DIR/'locked_test_summary.json').read_text(encoding='utf-8'))
display(pd.read_csv(OUTPUT_DIR/'locked_test_metrics.csv'))
display(pd.read_csv(OUTPUT_DIR/'locked_test_class_metrics.csv'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['configuration_locked_from_validation'] is True
assert summary['used_test_for_selection'] is False
print('고정 test 게이트 통과:', summary['locked_test_gate_passed'])
print('배포 모델 변경:', summary['deployment_model_changed'])


## 결과 ZIP 다운로드

다운로드된 `wm811k_robust_ensemble_locked_test_results.zip`을 프로젝트의 `결과물/wm811k/colab_가져오기/` 폴더에 압축을 풀지 않고 넣으세요.


In [ ]:
EXPORT_DIR = Path('/content/wm811k_robust_ensemble_locked_test_results')
if EXPORT_DIR.exists(): shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir()
for name in ('locked_test_summary.json', 'locked_test_metrics.csv', 'locked_test_class_metrics.csv', 'locked_test_confusion_long.csv'):
    shutil.copy2(OUTPUT_DIR/name, EXPORT_DIR/name)
zip_path = Path(shutil.make_archive(str(EXPORT_DIR), 'zip', root_dir=EXPORT_DIR))
print('다운로드:', zip_path)
files.download(str(zip_path))
